In [0]:
# Notebook autonome : il porte ses propres %run et se lance seul.
# Relancés par main_translations, ils sont sans effet de bord (idempotents).

In [0]:
# Périmètre du projet : 4 langues.
# Ce dictionnaire fait deux choses à la fois :
#   - il DÉCLARE les langues supportées (celles absentes ici sont hors périmètre) ;
#   - il fait correspondre le code métier au code langue ISO 639-1 renvoyé par
#     USERCULTURE(), qui n'est pas le même pour le tchèque.
# Ajouter une langue au projet = ajouter une ligne ici, rien d'autre.
CULTURE_MAP = {
    "FR": "fr",
    "EN": "en",
    "RO": "ro",
    "CZ": "cs",   # tchèque : ISO 639-1 = cs, la source porte le code pays CZ
}

# Hors périmètre pour l'instant : PL (polonais) et UA (ukrainien), présents dans
# parameters_languages mais non traités par le projet. Attention si on les ajoute :
# l'ukrainien est "uk" en ISO 639-1, pas "ua".

# Langue de repli quand la culture de l'utilisateur n'est pas dans le périmètre
FALLBACK_LANGUAGE_CODE = "en"

culture_expr = F.create_map([F.lit(x) for x in sum(CULTURE_MAP.items(), ())])

In [0]:
dim_trad_language = (
    spark.table(f"{source_catalog}.parameters_languages")
    .filter(F.col("deleted") == False)
    .withColumn("culture_code", culture_expr[F.col("code")])
    # Restreint au périmètre du projet : une langue absente de CULTURE_MAP a un
    # culture_code null et n'entre pas dans le modèle. C'est ce filtre qui pilote
    # tout le reste : le produit cartésien de build_translations porte sur
    # dim_trad_language, donc les tables de traduction ne contiendront que ces langues.
    .filter(F.col("culture_code").isNotNull())
    .select(
        F.col("id_parameter_language").alias("language"),
        F.col("code"),
        F.col("name").alias("language_label"),
        "culture_code",
    )
)

In [0]:
# Langues présentes en base mais hors périmètre du projet : informatif, pas une alerte.
langues_hors_perimetre = (
    spark.table(f"{source_catalog}.parameters_languages")
    .filter(F.col("deleted") == False)
    .filter(~F.col("code").isin(list(CULTURE_MAP.keys())))
    .select("id_parameter_language", "code", "name")
)

if verbose_mode == 'debug':
    print(f"Périmètre du projet : {len(CULTURE_MAP)} langues -> {list(CULTURE_MAP.keys())}")
    display(dim_trad_language)

    if langues_hors_perimetre.count() > 0:
        print("Langues présentes en base mais hors périmètre (non traduites) :")
        display(langues_hors_perimetre)

Périmètre du projet : 4 langues -> ['FR', 'EN', 'RO', 'CZ']


language,code,language_label,culture_code
1,FR,Français,fr
2,EN,English,en
5,RO,Romanian,ro
6,CZ,Czech,cs


Langues présentes en base mais hors périmètre (non traduites) :


id_parameter_language,code,name
3,PL,Polski
4,UA,українська


In [0]:
# La langue de repli doit exister, sinon le garde-fou du rôle RLS ne protège rien
if dim_trad_language.filter(F.col("culture_code") == FALLBACK_LANGUAGE_CODE).count() == 0:
    raise ValueError(
        f"La langue de repli '{FALLBACK_LANGUAGE_CODE}' est absente de dim_trad_language : "
        "le rôle RLS renverrait un rapport vide pour toute culture non reconnue."
    )

Import dim_trad_language

In [0]:
current_process = "dim_trad_language"

In [0]:
# Cible : le schéma commun, composé comme dans dim_site.
target_dim_trad_language = current_catalog + "." + common_schema + "." + current_process
print(target_dim_trad_language)

mal_maite_bi_preprd.common.dim_trad_language


In [0]:
all_columns = dim_trad_language.columns
display(all_columns)

_1
language
code
language_label
culture_code


In [0]:
# define the primary key
primary_key = ['language']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    print(additional_columns)

['code', 'language_label', 'culture_code']


In [0]:
ensure_delta_table(dim_trad_language, target_dim_trad_language)

# mode="full" imposé, et non execution_mode : dim_trad_language est entièrement
# redérivée à chaque run (filtrée sur CULTURE_MAP). Une langue retirée du
# périmètre doit disparaître du modèle, sinon le RLS continuerait de la proposer.
handle_table_update(
    dim_trad_language,
    target_dim_trad_language,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full"
)

Table mal_maite_bi_preprd.common.dim_trad_language créée (schéma seul, 0 ligne).
Table mal_maite_bi_preprd.common.dim_trad_language has been fully replaced (delete + insert).
Number of rows inserted: 4
